# 🔬 Quantum ML for Medical Imaging
### Quantum for Healthcare — Quantum for Humanity

This notebook applies **Quantum Support Vector Machines (QSVC)** to medical image classification — detecting tumours and disease markers in imaging data.

**Application:** Classifying malignant vs benign tissue from radiological features  
**Learning source:** [IBM Quantum Learning — Machine Learning](https://learning.quantum.ibm.com)

---

## Clinical Motivation

Medical imaging (MRI, CT, X-ray, Pathology slides) generates enormous datasets. Accurate, fast classification of disease vs. no-disease can:
- **Save lives** through earlier diagnosis
- **Reduce radiologist workload** in resource-scarce settings
- **Enable screening** in rural and low-income areas with limited specialists

Quantum kernel methods may offer advantages on structured medical data in the post-NISQ era.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute

print('✅ Imports successful')

## Step 1: Load Breast Cancer Dataset (Wisconsin Diagnostic)

The Wisconsin Diagnostic Breast Cancer dataset contains 30 features derived from digitised FNA (fine needle aspirate) images of breast mass tissue.

In [ ]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target  # 0=malignant, 1=benign

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: Malignant (0): {(y==0).sum()} | Benign (1): {(y==1).sum()}')
print(f'\nFeatures (first 5): {data.feature_names[:5].tolist()}')
print('...')

# Standardise
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to reduce to 4 features for quantum (4 qubits)
pca = PCA(n_components=4, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'\nPCA variance explained (4 components): {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Split
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Step 2: Visualise PCA Feature Space

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cls, label, color in [(0, 'Malignant', '#FF6B9D'), (1, 'Benign', '#8B5CF6')]:
    idx = y == cls
    axes[0].scatter(X_pca[idx, 0], X_pca[idx, 1], label=label, alpha=0.6, s=30, c=color)
    axes[1].scatter(X_pca[idx, 2], X_pca[idx, 3], label=label, alpha=0.6, s=30, c=color)

axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('PC1 vs PC2'); axes[0].legend()
axes[1].set_xlabel('PC3'); axes[1].set_ylabel('PC4')
axes[1].set_title('PC3 vs PC4'); axes[1].legend()

plt.suptitle('Breast Cancer Dataset — PCA Feature Space (4D → 2D projections)', fontsize=13)
plt.tight_layout(); plt.show()

## Step 3: Classical SVM Baseline

In [ ]:
classical_svm = SVC(kernel='rbf', C=10.0, probability=True, random_state=42)
classical_svm.fit(X_train, y_train)
y_pred_c = classical_svm.predict(X_test)
y_prob_c = classical_svm.predict_proba(X_test)[:, 1]

print('=== Classical RBF SVM ===')
print(classification_report(y_test, y_pred_c, target_names=['Malignant', 'Benign']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_c):.4f}')

## Step 4: Quantum Kernel SVM (QSVC)

In [ ]:
# Build 4-qubit ZZFeatureMap quantum kernel
feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='full')
print(f'Quantum circuit: {feature_map.num_qubits} qubits, depth {feature_map.depth()}')
feature_map.draw('mpl', style='clifford', fold=-1)

In [ ]:
# Quantum kernel estimation
sampler   = Sampler()
fidelity  = ComputeUncompute(sampler=sampler)
qkernel   = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

# Use a subset for simulation performance
n_train, n_test = 80, 40
qsvc = QSVC(quantum_kernel=qkernel, C=10.0)
qsvc.fit(X_train[:n_train], y_train[:n_train])
y_pred_q = qsvc.predict(X_test[:n_test])

print('\n=== Quantum Kernel SVM (QSVC) ===')
print(classification_report(y_test[:n_test], y_pred_q, target_names=['Malignant', 'Benign']))

## Step 5: ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_estimator(classical_svm, X_test, y_test, ax=ax, 
                                name='Classical RBF SVM', color='#8B5CF6')
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_title('ROC Curve — Breast Cancer Detection\nClassical SVM (Quantum comparison requires hardware)',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 🌍 Humanitarian Impact

| Application | Disease | Population Impact |
|---|---|---|
| Tumour classification | Breast cancer | 2.3M new cases/year globally |
| Chest X-ray analysis | TB, Pneumonia | 10M TB cases/year |
| Retinal scan analysis | Diabetic retinopathy | 100M at risk in India alone |
| Histopathology | Cervical cancer | 600K cases/year globally |

**Quantum + AI medical imaging** can bring specialist-level diagnosis to:
- Rural clinics with no radiologists
- Mobile screening vans in low-income communities
- Telemedicine platforms serving billions

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*